# 05 — Vectorization

**Repository:** `AI_Engineer-DL` | **Section:** `01_neural_networks_and_dl/logistic_regression`  
**File:** `notebooks/05_vectorization.ipynb`

> **Prerequisites:** `04_computation_graph_and_backprop.ipynb` — you need the gradient formulas  
> dZ=A−Y, dW=(1/m)·X·dZᵀ, db=(1/m)·sum(dZ) before implementing them in vectorised form.
>
> **Core Focus:** You eliminate every for-loop from logistic regression using NumPy broadcasting  
> and matrix operations, time the difference against a loop-based version, and understand  
> NumPy rank-1 array bugs. This is the last notebook in the logistic regression section.

## 1. What Is This?

The loop-based logistic regression you built in notebook 04 works correctly but is slow —  
it processes one training example at a time with an explicit Python for-loop.  
Vectorization replaces those loops with NumPy matrix operations that run in compiled C/SIMD code,  
often 100–300× faster. At the scale of modern datasets this difference is the  
boundary between "trains overnight" and "trains in minutes."

**Real-world connection — CloudyDrive:** Your YOLO model processes 32 frames simultaneously  
in each forward pass batch. The entire batch goes through every layer as one matrix multiply,  
not 32 sequential single-example passes. This is vectorization — the same idea you implement here,  
just extended to convolutional layers.

| Concept | What it is | Why it matters |
|---------|-----------|----------------|
| **Vectorization** | Replacing for-loops with matrix operations | 100–300× speedup on CPU; essential on GPU |
| **Broadcasting** | NumPy auto-expanding smaller arrays to match shapes | Enables Z = wᵀX + b without looping over m |
| **Rank-1 array** | Shape (n,) — neither row nor column | Source of silent shape bugs; always use (n,1) or (1,n) |
| **SIMD** | Single Instruction Multiple Data — CPU-level parallelism | Why NumPy ops are fast even without GPU |
| **Vectorised forward pass** | Z=wᵀX+b, A=σ(Z) — all m examples at once | Entire dataset in 2 lines |

## 2. The Math

### A — From loop to matrix: forward pass

**Loop version (slow):**
```
for i in range(m):
    z[i] = w.T @ x[:,i] + b
    a[i] = σ(z[i])
```

**Vectorised version (fast):**
$$Z = w^T X + b \qquad A = \sigma(Z)$$

where X has shape (nₓ, m), w has shape (nₓ, 1), b is a scalar.  
Result: Z and A each have shape (1, m) — all m examples computed simultaneously.

**Broadcasting:** b is a scalar but adds to every element of the (1, m) matrix wᵀX.  
NumPy expands b to shape (1, m) automatically.

---

### B — Vectorised backward pass

$$dZ = A - Y \quad \text{shape: } (1, m)$$
$$dW = \frac{1}{m} X \cdot dZ^T \quad \text{shape: } (n_x, 1)$$
$$db = \frac{1}{m} \sum dZ \quad \text{scalar}$$

No for-loop over m — all m gradients computed and averaged in three lines.

---

### C — The Rank-1 Array Bug

`np.random.randn(5)` creates shape **(5,)** — a rank-1 array.  
`.T` on a rank-1 array returns the **same shape** (5,). Transposing does nothing.  
This means `a.T @ a` returns a scalar instead of a (5,5) matrix — a silent, fatal bug.

**Rule:** Always use `np.random.randn(5, 1)` for column vectors and `np.random.randn(1, 5)` for rows.  
Add assertions to catch shape bugs early:
```python
assert w.shape == (nx, 1), f"Expected ({nx},1), got {w.shape}"
```

---

### D — Broadcasting rules

Two arrays are broadcastable when, comparing dimensions right-to-left,  
each pair is either equal, or one of them is 1.

| Operation | Shapes | Result |
|-----------|--------|--------|
| Z = wᵀX + b | (1,m) + scalar | (1,m) scalar broadcasts |
| A − Y | (1,m) − (1,m) | (1,m) same shape |
| X @ dZᵀ | (nₓ,m) @ (m,1) | (nₓ,1) |
| (3,4) + (1,4) | row broadcast | (3,4) |
| (3,4) + (3,1) | col broadcast | (3,4) |

In [ ]:
import numpy as np
import time
import matplotlib.pyplot as plt

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

## 3. Build From Scratch

### 3.1 — The rank-1 array bug (see it, then fix it)

Before writing any fast code, you must understand the most common NumPy bug  
in deep learning implementations.

> 🔮 **Predict before you run:** What do you expect `a.shape`, `a.T.shape`,  
> and `(a.T @ a).shape` to be if `a = np.random.randn(5)`?  
> Write your predictions before running.

In [ ]:
# YOUR CODE HERE
# ── Task ─────────────────────────────────────────────────────────────────────
# 1. Create a rank-1 array: a = np.random.randn(5)
# 2. Print a.shape, a.T.shape, and (a.T @ a) — show the bug.
# 3. Create a proper column vector: a_col = a.reshape(5, 1)  OR  np.random.randn(5, 1)
# 4. Print a_col.shape, a_col.T.shape, and (a_col.T @ a_col).shape.
#    Show that now you get a (1,1) scalar result from a proper inner product,
#    and (5,5) from the outer product a_col @ a_col.T.
# 5. Write an assert that would catch the bug: assert a_col.shape == (5, 1)
# ─────────────────────────────────────────────────────────────────────────────


> 💡 **Reflect:** The shape (5,) looks harmless but causes silent errors in matrix multiply.  
> The assertion `assert w.shape == (nx, 1)` costs one line and saves hours of debugging.  
> Add it after every initialisation in real code.

---

### 3.2 — Vectorised forward pass (zero for-loops)

Compute Z and A for all m examples simultaneously using matrix operations.

> 🔮 **Predict before you run:** If X has shape (nₓ, m) and w has shape (nₓ, 1),  
> what is the shape of wᵀX? And after adding scalar b? Write both shapes before coding.

In [ ]:
# YOUR CODE HERE
# ── Task ─────────────────────────────────────────────────────────────────────
# Set up the data:
#   np.random.seed(42)
#   nx, m = 5, 100
#   X = np.random.randn(nx, m)
#   Y = np.random.randint(0, 2, (1, m))
#   w = np.random.randn(nx, 1) * 0.01
#   b = 0.0
#
# 1. Compute Z = w.T @ X + b   (NO for-loops)
# 2. Compute A = sigmoid(Z)
# 3. Assert: Z.shape == (1, m) and A.shape == (1, m)
# 4. Print Z.shape, A.shape, and a sample: A[0, :5]
# ─────────────────────────────────────────────────────────────────────────────


---

### 3.3 — Vectorised backward pass (zero for-loops)

Compute all gradients at once: dZ, dW, db — for all m examples, averaged.

> 🔮 **Predict before you run:** X has shape (nₓ, m) and dZ has shape (1, m).  
> What operation gives dW of shape (nₓ, 1)? Think about which arrays multiply  
> and in what order.

In [ ]:
# YOUR CODE HERE
# ── Task ─────────────────────────────────────────────────────────────────────
# Using Z and A from 3.2:
# 1. dZ = A - Y              shape: (1, m)
# 2. dW = (1/m) * X @ dZ.T  shape: (nx, 1)  — do NOT loop over examples
# 3. db = (1/m) * np.sum(dZ) scalar
#
# Assert:
#   dZ.shape == (1, m)
#   dW.shape == (nx, 1)
#   isinstance(db, float) or db.shape == ()  — db is scalar
#
# Print all shapes and the first 3 elements of dW.
# ─────────────────────────────────────────────────────────────────────────────


---

### 3.4 — Speed comparison: vectorised vs loop

Time both approaches on large data to see the practical speedup.

> 🔮 **Predict before you run:** For nx=1000 and m=10000, how much faster do you  
> expect the vectorised version to be? Guess a multiplier before running.

In [ ]:
# YOUR CODE HERE
# ── Task ─────────────────────────────────────────────────────────────────────
# Compare vectorised forward pass vs loop-based forward pass.
# Setup:
#   np.random.seed(0)
#   nx_large, m_large = 1000, 10000
#   X_large = np.random.randn(nx_large, m_large)
#   w_large = np.random.randn(nx_large, 1)
#   b_large = 0.5
#
# 1. VECTORISED: Z_vec = w_large.T @ X_large + b_large  — time with time.time()
#
# 2. LOOP: initialise Z_loop = np.zeros((1, m_large))
#    for i in range(m_large):
#        Z_loop[0, i] = np.dot(w_large.T, X_large[:, i]) + b_large
#    time this loop.
#
# 3. Verify both give the same result: np.allclose(Z_vec, Z_loop)
# 4. Print: vectorised time, loop time, speedup factor.
# ─────────────────────────────────────────────────────────────────────────────


> 💡 **Reflect:** The speedup you measured is real and consistent.  
> In a network with 50 layers processing 256-image batches, this difference compounds —  
> vectorisation is not optional, it is the foundation modern deep learning is built on.

---

### 3.5 — Complete vectorised logistic regression: one iteration

Put everything together into a single function that runs one gradient descent step  
for all m examples with zero explicit for-loops.

> 🔮 **Predict before you run:** After one step starting from w=zeros and b=0,  
> should the cost decrease, increase, or stay the same? Why?

In [ ]:
# YOUR CODE HERE
# ── Task ─────────────────────────────────────────────────────────────────────
# Implement lr_step(X, Y, w, b, alpha) — one complete gradient descent step.
# Returns: updated (w, b, cost_before)
# Zero for-loops allowed. Use only: @, np.sum, sigmoid, np.log, np.clip.
#
# Test: seed 3, nx=4, m=20, alpha=0.1
#   Run 500 iterations. Store cost at each step.
#   Print: initial cost, final cost, final w[:3], final b.
# ─────────────────────────────────────────────────────────────────────────────

def lr_step(X, Y, w, b, alpha):
    """
    One vectorised gradient descent step for logistic regression.
    No for-loops. Returns updated (w, b, cost).
    """
    pass


> ✅ **Self-check:** Before moving on:
> 1. Write the three vectorised backward-pass equations from memory.
> 2. What is a rank-1 array and why is it dangerous? Name one specific failure mode.
> 3. Why does broadcasting work when adding (1, m) + scalar but NOT when shapes mismatch?

## 4. Library Version

`sklearn.linear_model.LogisticRegression` is the production version.  
Train it on the same data and compare weights — they should converge to similar values.

In [ ]:
from sklearn.linear_model import LogisticRegression

np.random.seed(3)
nx_t, m_t = 4, 20
X_t = np.random.randn(nx_t, m_t)
Y_t = np.random.randint(0, 2, (1, m_t))

# sklearn expects (m, nx) and (m,)
clf = LogisticRegression(C=1e6, max_iter=10000, solver='lbfgs').fit(X_t.T, Y_t.flatten())

# Our final w and b after 500 steps (assumes lr_step loop was run above)
# If not run yet, run 500 iterations here:
w_v = np.zeros((nx_t, 1));  b_v = 0.0; costs_v = []
for _ in range(500):
    w_v, b_v, c = lr_step(X_t, Y_t, w_v, b_v, alpha=0.1)
    costs_v.append(c)

print("Our w (first 3):    ", w_v[:3].flatten().round(4))
print("sklearn coef (first 3):", clf.coef_[0][:3].round(4))
print("Our b:     ", round(float(b_v), 4))
print("sklearn intercept:", round(clf.intercept_[0], 4))
print("(Values won't be identical — different optimisation paths, same optimum.")

## 5. Visualisation

### 5.1 — Speedup chart and cost curve

In [ ]:
import os
os.makedirs('../images', exist_ok=True)

# Speedup bar chart (use your measured times here)
# Replace with your actual times from 3.4
# These are placeholders — your code produces the real values
labels   = ['Vectorised', 'Loop']
# You will replace these with your measured values from Task 3.4:
# times = [t_vec, t_loop]
times    = [0.002, 0.8]   # illustrative placeholder — replace with your measured values

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left: time comparison
bars = axes[0].bar(labels, times, color=['steelblue', 'coral'], width=0.4, edgecolor='white')
for bar, t in zip(bars, times):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                 f'{t:.3f}s', ha='center', va='bottom', fontsize=11, fontweight='bold')
axes[0].set_ylabel('Time (seconds)', fontsize=12)
axes[0].set_title('Vectorised vs Loop: Forward Pass Time', fontsize=11, fontweight='bold')
axes[0].grid(axis='y', alpha=0.3)

# Right: cost curve
axes[1].plot(costs_v, color='steelblue', lw=2)
axes[1].set_xlabel('Iteration', fontsize=12)
axes[1].set_ylabel('Cost J(w, b)', fontsize=12)
axes[1].set_title('Vectorised LR: Cost over 500 Iterations', fontsize=11, fontweight='bold')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('../images/05_vectorization.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → ../images/05_vectorization.png')

## 6. Revision Corner

---

### One-sentence definition

> Vectorization replaces Python for-loops over training examples with NumPy matrix operations  
> that run in compiled, SIMD-parallel code, giving 100–300× speedups and enabling  
> the scale of modern deep learning.

---

### Why it exists — what problem does it solve?

Python is interpreted and loops are slow. Modern CPUs can execute the same operation  
on many data elements simultaneously (SIMD) — but only if data is laid out as arrays,  
not accessed one element at a time via a loop. NumPy operations pass the entire array  
to optimised C code, enabling massive parallelism. Without vectorization, training  
a model on 1 million examples would take weeks instead of hours.

---

### Interview gotchas

| Question | Common mistake | The reality |
|----------|---------------|-------------|
| **What is a rank-1 array and why is it dangerous?** | "It's just a 1D array" | Shape (n,) means `.T` does nothing and `a @ a` gives a scalar, not a matrix. Always use (n,1) column vectors. The bug is silent — no error is thrown. |
| **What does broadcasting do?** | "It copies data" | It virtually expands a smaller array to match a larger one without actually copying memory. (1,m) + scalar adds the scalar to every element. (nₓ,m) + (nₓ,1) adds each row element to all m columns. |
| **Write the vectorised backward pass equations.** | Forget the (1/m) | dZ=A−Y, dW=(1/m)·X·dZᵀ, db=(1/m)·np.sum(dZ). Forgetting 1/m makes gradients scale with m, breaking the learning rate. |
| **Why use keepdims=True in np.sum(dZ)?** | Not needed | Without keepdims, np.sum(dZ, axis=1) on shape (1,m) gives shape (1,) not (1,1). This causes a broadcast mismatch in the update step. Use keepdims=True to preserve shape. |
| **Can you have two for-loops in an LR implementation?** | "Yes, one over m" | No. A correct vectorised implementation has zero explicit loops: one matrix multiply covers all m examples in both forward and backward pass. |

## 7. Exercises

Write solutions in `exercises/05_vectorization_exN_label.py`.

---

### ⭐ Exercise 1 — Rank-1 array trap

Demonstrate the rank-1 bug in 4 steps:  
1. Create `a = np.random.randn(6)`. Show that `a.T.shape == a.shape`.  
2. Show that `a @ a` gives a scalar, not a (6,6) matrix.  
3. Fix it: `a_col = a.reshape(6, 1)`. Verify `a_col.T.shape == (1, 6)`.  
4. Verify `(a_col @ a_col.T).shape == (6, 6)` (outer product) and `(a_col.T @ a_col).shape == (1, 1)` (inner product).  
Use np.isclose or shape assertions for all checks.

---

### ⭐⭐ Exercise 2 — Vectorised forward+backward, verified against loop

Seed 7, nx=3, m=8, w=randn(3,1)*0.01, b=0.0.  
1. Vectorised: compute Z, A, dZ, dW, db in zero loops.  
2. Loop: for i in range(m), compute z_i, a_i, dz_i, dw_i, db_i — accumulate and average.  
3. Verify: np.allclose(dW_vec, dW_loop) and np.isclose(db_vec, db_loop).  
Print both versions and the allclose result.

---

### ⭐⭐⭐ Exercise 3 — Full vectorised logistic regression, end-to-end

**Scenario:** Cat vs Non-Cat classifier — the classic Week 2 assignment problem.  
Seed 42. m_train=200, m_test=50, nx=20 (proxy for real image features).  
Generate: X_train = randn(20, 200), Y_train = randint(0,2,(1,200)).  
           X_test  = randn(20, 50),  Y_test  = randint(0,2,(1,50)).

Implement `train_lr(X, Y, alpha, num_iterations)` that:  
- Initialises w=zeros, b=0.  
- Runs the vectorised loop for num_iterations.  
- Returns (w, b, costs).  

Implement `evaluate(X, Y, w, b)` that returns accuracy.  

Train for 2000 iterations with α=0.005.  
Print: train accuracy, test accuracy, final cost.  
Plot cost curve. Save to `../images/05_vectorization_exercise.png`.  
Comment: why might the test accuracy be lower than train accuracy, even here on random data?